In [3]:
import pandas as pd
import numpy as np

# ML tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Pipeline
from sklearn.compose import make_column_transformer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# -------------------------------
# 1. Load Data
# -------------------------------
df = pd.read_csv("Cleaned_Dataset.csv")

print("Dataset Shape:", df.shape)

# -------------------------------
# 2. Feature Engineering
# -------------------------------
df["SalaryGrowth"] = df["PercentSalaryHike"] / (df["YearsAtCompany"] + 1)
df["PromotionDelay"] = df["YearsSinceLastPromotion"] / (df["YearsAtCompany"] + 1)
df["WorkPressure"] = df["OverTime"] * (4 - df["WorkLifeBalance"])
df["CareerStagnation"] = (df["YearsInCurrentRole"] + df["YearsSinceLastPromotion"]) / (df["TotalWorkingYears"] + 1)
df["Stability"] = df["YearsAtCompany"] / (df["TotalWorkingYears"] + 1)
df["IncomePerLevel"] = df["MonthlyIncome"] / (df["JobLevel"] + 1)
df["CommuteStress"] = df["DistanceFromHome"] * df["OverTime"]

# Drop original columns
df.drop([
    "PercentSalaryHike", "YearsAtCompany", "YearsSinceLastPromotion",
    "OverTime", "WorkLifeBalance", "YearsInCurrentRole",
    "TotalWorkingYears", "MonthlyIncome", "JobLevel", "DistanceFromHome"
], axis=1, inplace=True)

# -------------------------------
# 3. Split Data
# -------------------------------
X = df.drop("Attrition", axis=1)
y = df["Attrition"]

cat_cols = ["BusinessTravel", "Department", "EducationField", "Gender", "JobRole", "MaritalStatus"]
num_cols = X.columns.difference(cat_cols)

col_trans = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'), cat_cols),
    (StandardScaler(), num_cols)
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
# 4. Models
# -------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_split=10, min_samples_leaf=5, class_weight='balanced'
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, class_weight='balanced'
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=5
    ),
    
    "SVM": SVC(probability=True, class_weight='balanced'),
    
    "KNN": KNeighborsClassifier(n_neighbors=7, weights='distance'),
    
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, random_state=42
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, class_weight='balanced'
    )
}

# -------------------------------
# 5. Training & Evaluation
# -------------------------------
results = []

for name, model in models.items():
    
    pipe = Pipeline(steps=[
        ("preprocessor", col_trans),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)
    y_test_prob = pipe.predict_proba(X_test)[:, 1]
    
    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    prec = precision_score(y_test, y_test_pred)
    rec = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    auc = roc_auc_score(y_test, y_test_prob)
    
    results.append([
        name, train_acc, test_acc, prec, rec, f1, auc
    ])

# -------------------------------
# 6. Results
# -------------------------------
results_df = pd.DataFrame(results, columns=[
    "Model", "Train Accuracy", "Test Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"
])

results_df = results_df.sort_values(by="F1 Score", ascending=False)

print("\nFinal Model Comparison:\n")
print(results_df)

results_df.to_csv("model_comparison.csv", index=False)
print("\n✅ Model comparison saved to model_comparison.csv")

Dataset Shape: (1470, 33)
[LightGBM] [Info] Number of positive: 986, number of negative: 986
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001493 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8481
[LightGBM] [Info] Number of data points in the train set: 1972, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [4]:
import pandas as pd

df2 = pd.read_csv(r'C:\Users\hp\Desktop\Employee Attrition Model\model_comparison.csv')

print(df2.head())

                 Model  Train Accuracy  Test Accuracy  Precision    Recall  \
0  Logistic Regression        0.793367       0.782313   0.392405  0.659574   
1                  SVM        0.961735       0.850340   0.538462  0.446809   
2              XGBoost        0.969388       0.860544   0.625000  0.319149   
3             LightGBM        0.963435       0.860544   0.625000  0.319149   
4    Gradient Boosting        0.996599       0.853741   0.576923  0.319149   

   F1 Score   ROC-AUC  
0  0.492063  0.793350  
1  0.488372  0.743819  
2  0.422535  0.770695  
3  0.422535  0.766733  
4  0.410959  0.767594  
